# Medical Image Analyzer — Multi-Model VLM Comparison (Google Colab T4)

This notebook benchmarks **4 medical-image-trained Vision-Language Models (VLMs)** sequentially in 4-bit quantization on a **Google Colab T4 GPU (15 GB VRAM)**:
1. **MedGemma 1.5 (4B)** — `google/medgemma-1.5-4b-it` *(gated — requires `HF_TOKEN`)*
2. **LLaVA-Med v1.5 (7B)** — `chaoyinshe/llava-med-v1.5-mistral-7b-hf` *(ungated)*
3. **MedVLM-R1 (2B)** — `JZPeterPan/MedVLM-R1` *(ungated, GRPO medical reasoning)*
4. **Qwen2.5-VL-MedSFT (3B)** — `CAIR-HKISI/Qwen2.5-VL-3B-MedSFT` *(ungated)*

**Dataset (150 Uncontaminated Clinical Images):**
- 75 stratified single-image cases from **`TsinghuaC3I/MedXpertQA` (`MM`, 2025)**
- 75 recent clinical image challenges from **`OctoMed/NEJM-Image-Challenge`** (with **Human Clinician Crowd Accuracy %**)

> **Before running:** Set **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# 1. Check GPU & Mount Google Drive for persistent checkpoints and results
!nvidia-smi

import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_WORK_DIR = Path('/content/drive/MyDrive/Medical_Image_Analyzer_Project')
DRIVE_WORK_DIR.mkdir(parents=True, exist_ok=True)
print('Google Drive workspace:', DRIVE_WORK_DIR)

In [ ]:
# 2. Clone / Update project repository and install dependencies
REPO_URL = 'https://github.com/JapjeetSingh17/Medical-Image-Analysis-Multi-Model-Comparison.git'
REPO_DIR = Path('/content/Medical-Image-Analysis-Multi-Model-Comparison')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd /content/Medical-Image-Analysis-Multi-Model-Comparison
!pip install -q -r requirements-colab.txt

In [ ]:
# 3. Configure Hugging Face Token (needed for gated google/medgemma-1.5-4b-it)
# Store your token in Colab Secrets as HF_TOKEN, or paste when prompted.
import os
from getpass import getpass

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

if not hf_token:
    hf_token = getpass('Enter Hugging Face Token (or press Enter to skip gated MedGemma 1.5): ').strip()

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN configured.')
else:
    print('No HF_TOKEN provided; MedGemma 1.5 will be skipped and the 3 ungated models will run.')

In [ ]:
# 4. Build the 150-image uncontaminated dataset (saved to Google Drive so it persists)
DATA_DIR = DRIVE_WORK_DIR / 'data' / 'benchmark_150'
RESULTS_DIR = DRIVE_WORK_DIR / 'results'

if not (DATA_DIR / 'manifest.csv').exists():
    !python -m src.prepare_dataset --output-dir "{DATA_DIR}" --medxpert-count 75 --nejm-count 75
else:
    print('Found existing dataset manifest at:', DATA_DIR / 'manifest.csv')

import pandas as pd
manifest_df = pd.read_csv(DATA_DIR / 'manifest.csv')
display(manifest_df[['case_id', 'source', 'domain', 'medical_task', 'case_date']].head(10))
print('Total cases:', len(manifest_df))

In [ ]:
# 5A. Quick Smoke Test (6 images across all models) — verifies VRAM & pipeline before full run
!python run_benchmark.py --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}_smoke" --smoke-test

In [ ]:
# 5B. Full 150-Image Benchmark (Checkpointed to Drive — safe to re-run to resume)
!python run_benchmark.py --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}" --full

In [ ]:
# 6. Display Ranked Leaderboard & Percentage Graphs Inline
from IPython.display import Image as IPyImage, Markdown, display

report_path = RESULTS_DIR / 'report.md'
if report_path.exists():
    display(Markdown(report_path.read_text()))

for chart_file in sorted((RESULTS_DIR / 'charts').glob('*.png')):
    print(chart_file.name)
    display(IPyImage(filename=str(chart_file)))

In [ ]:
# 7. Interactive Live Presentation Demo (Gradio with 3 Preloaded Clinical Cases + Upload)
# Generates a public gradio.live link you can present in class!
!python -m src.demo_app --manifest "{DATA_DIR}/manifest.csv" --launch-gradio

In [ ]:
# 8. Package Charts, Report, and Static Backup Demo Cards for Download
import shutil
from google.colab import files

zip_base = '/content/medical_vlm_results'
archive_path = shutil.make_archive(zip_base, 'zip', root_dir=str(RESULTS_DIR))
print('Created archive:', archive_path)
files.download(archive_path)